In [1]:
# Run in terminal if not already installed: pip install ultralytics

from pathlib import Path
import yaml

DATASET_ROOT = Path("..") / "smartvision_dataset"
DETECTION_DIR = DATASET_ROOT / "detection"

with open(DETECTION_DIR / "data.yaml", "r") as f:
    data_yaml_content = f.read()

print(data_yaml_content)

# SmartVision Dataset - YOLOv8 Configuration
path: /content/smartvision_dataset/detection
train: images
val: images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: truck
  7: traffic light
  8: stop sign
  9: bench
  10: bird
  11: cat
  12: dog
  13: horse
  14: cow
  15: elephant
  16: bottle
  17: cup
  18: bowl
  19: pizza
  20: cake
  21: chair
  22: couch
  23: potted plant
  24: bed

nc: 25



In [2]:
import shutil
import random

random.seed(42)

DETECTION_IMAGES = DETECTION_DIR / "images"
DETECTION_LABELS = DETECTION_DIR / "labels"

# Get all image files that have a matching label file
image_files = list(DETECTION_IMAGES.glob("*.jpg")) + list(DETECTION_IMAGES.glob("*.png"))
paired_files = []
for img in image_files:
    label = DETECTION_LABELS / (img.stem + ".txt")
    if label.exists():
        paired_files.append((img, label))

print(f"Total paired image/label files: {len(paired_files)}")

random.shuffle(paired_files)
n_total = len(paired_files)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

splits = {
    "train": paired_files[:n_train],
    "val": paired_files[n_train:n_train + n_val],
    "test": paired_files[n_train + n_val:]
}

for split_name, files in splits.items():
    (DETECTION_IMAGES / split_name).mkdir(parents=True, exist_ok=True)
    (DETECTION_LABELS / split_name).mkdir(parents=True, exist_ok=True)
    for img, label in files:
        shutil.copy(img, DETECTION_IMAGES / split_name / img.name)
        shutil.copy(label, DETECTION_LABELS / split_name / label.name)
    print(f"{split_name}: {len(files)} images")

Total paired image/label files: 2125
train: 1487 images
val: 318 images
test: 320 images


In [3]:
import os

# Use absolute path for reliability with YOLO training
absolute_detection_path = str(DETECTION_DIR.resolve())

new_data_yaml = {
    "path": absolute_detection_path,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate([
        "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
        "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
        "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake",
        "chair", "couch", "potted plant", "bed"
    ])},
    "nc": 25
}

with open(DETECTION_DIR / "data.yaml", "w") as f:
    yaml.dump(new_data_yaml, f, default_flow_style=False, sort_keys=False)

# Confirm it wrote correctly
with open(DETECTION_DIR / "data.yaml", "r") as f:
    print(f.read())

path: C:\Users\Admin\Documents\SmartVision\smartvision_dataset\detection
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: truck
  7: traffic light
  8: stop sign
  9: bench
  10: bird
  11: cat
  12: dog
  13: horse
  14: cow
  15: elephant
  16: bottle
  17: cup
  18: bowl
  19: pizza
  20: cake
  21: chair
  22: couch
  23: potted plant
  24: bed
nc: 25



In [4]:
from ultralytics import YOLO

# Load the smallest/fastest YOLOv8 variant (nano) — best choice for CPU training
model = YOLO("yolov8n.pt")  # auto-downloads pretrained COCO weights on first run

# Quick 1-epoch trial run to measure timing, using a smaller image size for CPU speed
results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=1,
    imgsz=320,
    batch=8,
    device="cpu",
    project="../models/yolo_runs",
    name="trial",
    exist_ok=True
)

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Admin\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1

In [5]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device="cpu",
    project="../models/yolo_runs",
    name="smartvision_yolov8n",
    exist_ok=True,
    patience=10  # early stopping if mAP plateaus
)

Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smartvision_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimiz

In [6]:
# Find any .cache files Ultralytics created
cache_files = list(DETECTION_DIR.rglob("*.cache"))
print("Found cache files:")
for f in cache_files:
    print(f)

# Delete them to force a fresh scan
for f in cache_files:
    f.unlink()
print("\nCache files deleted.")

# Verify actual file counts on disk right now
for split in ["train", "val", "test"]:
    img_dir = DETECTION_IMAGES / split
    lbl_dir = DETECTION_LABELS / split
    img_files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    lbl_files = list(lbl_dir.glob("*.txt"))
    print(f"{split}: {len(img_files)} images on disk, {len(lbl_files)} labels on disk")

Found cache files:
..\smartvision_dataset\detection\labels\train.cache
..\smartvision_dataset\detection\labels\val.cache

Cache files deleted.
train: 1487 images on disk, 1487 labels on disk
val: 318 images on disk, 318 labels on disk
test: 320 images on disk, 320 labels on disk


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=16,
    device="cpu",
    project="../models/yolo_runs",
    name="trial2",
    exist_ok=True
)

Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=trial2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

In [8]:
from PIL import Image as PILImage

def validate_split(split_name):
    img_dir = DETECTION_IMAGES / split_name
    lbl_dir = DETECTION_LABELS / split_name

    issues = {"unreadable_image": [], "empty_label": [], "bad_format": [],
              "bad_class_id": [], "bad_coords": [], "zero_size_box": []}
    valid_count = 0

    label_files = list(lbl_dir.glob("*.txt"))

    for lbl_file in label_files:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = img_dir / (lbl_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            issues["unreadable_image"].append(lbl_file.name)
            continue

        try:
            PILImage.open(img_path).verify()
        except Exception:
            issues["unreadable_image"].append(lbl_file.name)
            continue

        with open(lbl_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        if len(lines) == 0:
            issues["empty_label"].append(lbl_file.name)
            continue

        file_valid = True
        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                issues["bad_format"].append(lbl_file.name)
                file_valid = False
                break
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            if not (0 <= class_id <= 24):
                issues["bad_class_id"].append(lbl_file.name)
                file_valid = False
                break
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                issues["bad_coords"].append(lbl_file.name)
                file_valid = False
                break
            if w <= 0 or h <= 0:
                issues["zero_size_box"].append(lbl_file.name)
                file_valid = False
                break

        if file_valid:
            valid_count += 1

    print(f"\n=== {split_name} ===")
    print(f"Total label files: {len(label_files)}")
    print(f"Valid: {valid_count}")
    for issue_type, files in issues.items():
        if files:
            print(f"{issue_type}: {len(files)} (e.g. {files[:3]})")

validate_split("train")
validate_split("val")


=== train ===
Total label files: 1487
Valid: 342
bad_coords: 1145 (e.g. ['image_000002.txt', 'image_000005.txt', 'image_000010.txt'])

=== val ===
Total label files: 318
Valid: 65
bad_coords: 253 (e.g. ['image_000003.txt', 'image_000004.txt', 'image_000016.txt'])


In [9]:
def show_bad_lines(split_name, n_files=5):
    lbl_dir = DETECTION_LABELS / split_name
    shown = 0

    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        for line in lines:
            parts = line.split()
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                print(f"{lbl_file.name}: class={class_id} x={x} y={y} w={w} h={h}")
                shown += 1
                break
        if shown >= n_files:
            break

show_bad_lines("train", n_files=10)

image_000002.txt: class=0 x=0.485512 y=1.0362 w=0.344567 h=0.73556
image_000005.txt: class=0 x=0.485512 y=1.0362 w=0.344567 h=0.73556
image_000010.txt: class=12 x=0.376586 y=1.093815 w=0.559828 h=0.890681
image_000011.txt: class=12 x=0.376586 y=1.093815 w=0.559828 h=0.890681
image_000012.txt: class=12 x=0.376586 y=1.093815 w=0.559828 h=0.890681
image_000013.txt: class=12 x=0.376586 y=1.093815 w=0.559828 h=0.890681
image_000015.txt: class=12 x=0.376586 y=1.093815 w=0.559828 h=0.890681
image_000017.txt: class=0 x=1.05682 y=0.344627 w=0.83072 h=0.402693
image_000018.txt: class=0 x=1.05682 y=0.344627 w=0.83072 h=0.402693
image_000020.txt: class=0 x=1.05682 y=0.344627 w=0.83072 h=0.402693


In [10]:
def repair_labels(split_name):
    lbl_dir = DETECTION_LABELS / split_name
    repaired = 0
    now_empty = []

    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        valid_lines = []
        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            if 0 <= class_id <= 24 and 0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1 and w > 0 and h > 0:
                valid_lines.append(line)

        if len(valid_lines) != len(lines):
            repaired += 1
            with open(lbl_file, "w") as f:
                f.write("\n".join(valid_lines) + ("\n" if valid_lines else ""))

        if len(valid_lines) == 0:
            now_empty.append(lbl_file.name)

    print(f"{split_name}: repaired {repaired} files, {len(now_empty)} now have zero valid boxes")
    return now_empty

empty_train = repair_labels("train")
empty_val = repair_labels("val")
empty_test = repair_labels("test")

train: repaired 1145 files, 78 now have zero valid boxes
val: repaired 253 files, 16 now have zero valid boxes
test: repaired 239 files, 21 now have zero valid boxes


In [11]:
import os

def remove_empty_pairs(split_name, empty_label_names):
    img_dir = DETECTION_IMAGES / split_name
    lbl_dir = DETECTION_LABELS / split_name
    removed = 0

    for label_name in empty_label_names:
        stem = Path(label_name).stem
        lbl_path = lbl_dir / label_name
        if lbl_path.exists():
            lbl_path.unlink()
            removed += 1
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = img_dir / (stem + ext)
            if img_path.exists():
                img_path.unlink()

    print(f"{split_name}: removed {removed} empty image/label pairs")

remove_empty_pairs("train", empty_train)
remove_empty_pairs("val", empty_val)
remove_empty_pairs("test", empty_test)

# Confirm final counts
for split in ["train", "val", "test"]:
    img_count = len(list((DETECTION_IMAGES / split).glob("*.jpg"))) + len(list((DETECTION_IMAGES / split).glob("*.png")))
    lbl_count = len(list((DETECTION_LABELS / split).glob("*.txt")))
    print(f"{split}: {img_count} images, {lbl_count} labels remaining")

train: removed 78 empty image/label pairs
val: removed 16 empty image/label pairs
test: removed 21 empty image/label pairs
train: 1409 images, 1409 labels remaining
val: 302 images, 302 labels remaining
test: 299 images, 299 labels remaining


In [12]:
# Clear any cache again to be safe
cache_files = list(DETECTION_DIR.rglob("*.cache"))
for f in cache_files:
    f.unlink()
print(f"Cleared {len(cache_files)} cache files")

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device="cpu",
    project="../models/yolo_runs",
    name="smartvision_yolov8n_v2",
    exist_ok=True,
    patience=10
)

Cleared 2 cache files
Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smartvision_yolov8n_v2, nbs=64, nms=False, opset=None

In [14]:
model = YOLO("runs/models/yolo_runs/smartvision_yolov8n_v2/weights/best.pt")

results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device="cpu",
    project="../models/yolo_runs",
    name="smartvision_yolov8n_v3",
    exist_ok=True,
    patience=8
)

Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/models/yolo_runs/smartvision_yolov8n_v2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smartvision_yolov8n_v3, n

In [15]:
from ultralytics import YOLO

model = YOLO("runs/models/yolo_runs/smartvision_yolov8n_v3/weights/best.pt")

results = model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device="cpu",
    project="../models/yolo_runs",
    name="smartvision_yolov8n_v4",
    exist_ok=True,
    patience=8
)

Ultralytics 8.4.90  Python-3.14.3 torch-2.12.1+cpu CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\smartvision_dataset\detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/models/yolo_runs/smartvision_yolov8n_v3/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smartvision_yolov8n_v4, n

In [16]:
import json
import shutil

# Copy the final best weights to your models folder (used later by the Streamlit app)
shutil.copy(
    "runs/models/yolo_runs/smartvision_yolov8n_v4/weights/best.pt",
    "../models/yolov8_best.pt"
)
print("Saved to ../models/yolov8_best.pt")

# Save final metrics summary
final_metrics = {
    "model": "YOLOv8n",
    "mAP50": 0.752,
    "mAP50-95": 0.421,
    "precision": 0.913,
    "recall": 0.665,
    "epochs_trained": "30 + 20 + 20 (staged fine-tuning)",
    "inference_speed_ms": 19.8
}

with open("../results/yolo_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print("Saved metrics to ../results/yolo_metrics.json")

Saved to ../models/yolov8_best.pt
Saved metrics to ../results/yolo_metrics.json


In [2]:
from ultralytics import YOLO

model = YOLO("../models/yolov8_best.pt")

test_metrics = model.val(
    data=str(DETECTION_DIR / "data.yaml"),
    split="test",
    imgsz=640,
    device="cpu"
)

print(f"\nTest mAP50: {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test Precision: {test_metrics.box.mp:.4f}")
print(f"Test Recall: {test_metrics.box.mr:.4f}")

NameError: name 'DETECTION_DIR' is not defined

In [1]:
import random
import matplotlib.pyplot as plt

test_images_dir = DETECTION_IMAGES / "test"
sample_images = random.sample(list(test_images_dir.glob("*.jpg")), 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flatten(), sample_images):
    result = model.predict(source=str(img_path), imgsz=640, conf=0.5, verbose=False)[0]
    annotated = result.plot()  # returns BGR numpy array with boxes drawn
    annotated_rgb = annotated[..., ::-1]  # BGR to RGB
    ax.imshow(annotated_rgb)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../results/eda_plots/yolo_predictions_sample.png")
plt.show()

NameError: name 'DETECTION_IMAGES' is not defined